# AR Completion v2 — extended Adversarial Reactivity on the meta-evaluation sample

Supersedes `colab_ar_completion.ipynb` for the two $M^{*}$ explain-func metrics.
The v1 run had two protocol inconsistencies this notebook repairs:

| | v1 (Aug 2026) | **v2 (this notebook)** |
|---|---|---|
| Images | **first 12** of the test split (split order, not a random sample) | **the same seed-42 sample of 64 images** used by the main NR/AR meta-evaluation (`colab_full_run.ipynb`, cell 18) |
| Degradation levels | 5 (`linspace(0, 0.9, 5)`) — abs-Spearman granularity ~0.1 | **10** (`0.0, 0.1, ..., 0.9`) — finer AR resolution |
| Metrics | max_sensitivity, avg_sensitivity, random_logit | **max_sensitivity, random_logit** ($M^{*}$ members; avg_sensitivity is pruned from $M^{*}$ — re-enable via `INCLUDE_AVG_SENSITIVITY` for +~12 h) |
| Checkpoint unit | one (model, FAE, metric) cell | **one (model, FAE, metric, level, mini-batch-of-8) unit** (~4 s to ~3.5 min each) |

After this run, NR and AR for every $M^{*}$ metric rest on the **same 64-image
sample**, and the thesis can state a single meta-evaluation n.

**Cost (T4, scaled from v1 runtimes):** ~14–15 h total for the default two
metrics; the `occlusion` x `max_sensitivity` cells alone are ~10 h. Budget two
Colab Pro sessions. Every finished unit is appended to a local CSV immediately
and the whole file is snapshotted to Drive at least every 2 minutes (and after
every slow unit), so a disconnect loses at most ~2 minutes of work — re-run the
cells top-to-bottom and the run resumes where it left off.

**Prerequisites**
1. `git push` local master first (this notebook clones from GitHub and asserts
   the `make_degraded_explain_func` commit is present).
2. Runtime -> Change runtime type -> **T4 GPU**.
3. Drive: `thesis/weights/*.pth` and `thesis/data/{images,masks}/test/` as for
   the other notebooks.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results/checkpoints', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

## 2. Clone Repository (private — needs the GH_TOKEN Colab secret)

In [ ]:
import os

# Private repo: read the GitHub token from Colab secrets (key icon in the
# left sidebar -> add secret named GH_TOKEN with "Notebook access" enabled).
from google.colab import userdata
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    CLONE_URL = f'https://{GH_TOKEN}@github.com/dawkopagh/fae-metrics-master-thesis.git'
except Exception:
    print('No GH_TOKEN secret found - trying anonymous clone (works only if public).')
    CLONE_URL = 'https://github.com/dawkopagh/fae-metrics-master-thesis.git'

THESIS_ROOT = '/content/thesis-repo'
CODE_DIR    = f'{THESIS_ROOT}/fae-metrics-master-thesis'

if os.path.exists(THESIS_ROOT):
    %cd {THESIS_ROOT}
    !git pull {CLONE_URL} master
else:
    !git clone {CLONE_URL} {THESIS_ROOT}

%cd {CODE_DIR}
print(f'Working directory: {os.getcwd()}')
print('HEAD commit:', end=' ')
!git rev-parse HEAD

# Guard: the AR fix must be present.
from pathlib import Path
assert 'make_degraded_explain_func' in Path(
    'src/meta_evaluation/metaquantus_wrapper.py').read_text(), (
    'AR-fix commit missing - run `git push` locally first, then re-run this cell.')
print('AR fix present.')

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch
print(f'captum  {captum.__version__} | quantus {quantus.__version__} | '
      f'torch {torch.__version__} | cuda {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Weights from Drive (checksum-verified)

In [ ]:
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os, hashlib
os.makedirs('weights', exist_ok=True)
!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth

for path, expected in [('weights/resnet18_isic2017.pth', RESNET_SHA),
                       ('weights/squeezenet_isic2017.pth', SQUEEZENET_SHA)]:
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    status = 'OK' if actual == expected else 'MISMATCH!'
    print(f'{path}: {status}')
    assert actual == expected, f'Checksum mismatch for {path}'

## 5. Test Split from Drive (fallback: re-download)

In [ ]:
import os, sys
sys.path.insert(0, '.')

_needs_download = False
for split_dir in ['data/images/test', 'data/masks/test']:
    src = f'{DRIVE_THESIS}/{split_dir}'
    if os.path.exists(src):
        os.makedirs(split_dir, exist_ok=True)
        !cp -r {src}/* {split_dir}/
        n = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f'{split_dir}: {n} files (from Drive)')
    else:
        print(f'{split_dir}: not found on Drive — will download')
        _needs_download = True

if _needs_download:
    from src.data.download_isic import download_isic2017
    download_isic2017(dest_dir='data', skip_existing=True)
print('Data scaffold ready.')

## 6. Models, the seed-42 meta-evaluation sample, explainers, metrics

The image selection below is a **verbatim replication** of the main
meta-evaluation cell (`colab_full_run.ipynb`, cell 18, commit `89ebfcb`):
`np.random.default_rng(42).choice(600, size=64, replace=False)`, sorted, over
the test split in dataset order, with **targets = ResNet-18 predictions** for
both models — so the AR scores produced here pair exactly with the existing NR
scores in `meta_evaluation_reliability.csv`.

In [ ]:
import numpy as np, torch, quantus, pandas as pd

from src.models.classifiers import load_resnet18, load_squeezenet
from src.data.isic_dataset  import ISIC2017Dataset
from src.pipeline           import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import (
    _call_metric, make_degraded_explain_func,
)

# ----- Configuration ------------------------------------------------------
SEED       = 42
N_LEVELS   = 10      # degradation levels: linspace(0.0, 0.9, 10) = 0.0, 0.1, ..., 0.9
LEVEL_MAX  = 0.9     # frac=1.0 (all-zero attribution) is rejected by Quantus
MINI_BATCH = 8       # images per checkpoint unit (64 / 8 = 8 units per level)
INCLUDE_AVG_SENSITIVITY = False   # pruned from M*; True adds ~12 h
META_N_IMAGES = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth',   device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}

# ----- Seed-42 sample: verbatim from colab_full_run.ipynb cell 18 ---------
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
_N_FULL = len(dataset)
_rng = np.random.default_rng(42)
_sel = sorted(_rng.choice(_N_FULL, size=min(META_N_IMAGES, _N_FULL), replace=False).tolist())
samples = [dataset[i] for i in _sel]
N_IMAGES = len(samples)
print(f'Meta-eval sample: {N_IMAGES}/{_N_FULL} test images (seed 42).')

images  = [s['image'] for s in samples]
imgs_np = [img.detach().cpu().numpy() for img in images]
x_all   = np.stack(imgs_np)                       # (64, 3, 224, 224)

# Targets = ResNet-18 predictions (same convention as the main meta-eval).
_BATCH = 64
_img_t = torch.stack(images).to(device)
targets = []
with torch.no_grad():
    for _i in range(0, N_IMAGES, _BATCH):
        targets.extend(models['resnet18'](_img_t[_i:_i+_BATCH]).argmax(dim=1).tolist())
del _img_t
y_all = np.array(targets, dtype=np.int64)
print(f'Targets computed for {len(targets)} images.')

FAE_NAMES = ['integrated_gradients', 'saliency', 'gradcam',
             'deep_lift', 'guided_backprop', 'lrp', 'occlusion']
fae_funcs = {
    m: {name: _make_explain_func(name, models[m], m, device) for name in FAE_NAMES}
    for m in models
}

# Production metric configs — identical to the main meta-eval notebook.
metric_fns = {
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
}
if INCLUDE_AVG_SENSITIVITY:
    metric_fns['avg_sensitivity'] = quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True)

LEVELS = list(np.linspace(0.0, LEVEL_MAX, N_LEVELS))
N_BATCHES = N_IMAGES // MINI_BATCH
assert N_IMAGES % MINI_BATCH == 0, 'MINI_BATCH must divide N_IMAGES'
print(f'{len(models)} models x {len(FAE_NAMES)} FAE x {len(metric_fns)} metrics '
      f'x {N_LEVELS} levels x {N_BATCHES} mini-batches = '
      f'{len(models)*len(FAE_NAMES)*len(metric_fns)*N_LEVELS*N_BATCHES} units')

## 7. Run — per-unit checkpointing, Drive-durable

One **unit** = one (model, FAE, metric, level, mini-batch-of-8) metric call.
Each finished unit is appended to a local CSV immediately; the whole file is
snapshotted to Drive every `SYNC_SECONDS` (and after any unit slower than
60 s, so the long occlusion units are never lost). A fresh runtime restores
the Drive snapshot and resumes — completed units are never recomputed.

Each unit's RNG is seeded from its own coordinates
(`default_rng([SEED, model, fae, metric, level, batch])`), so results are
**identical whether the run completes in one session or ten** — resuming does
not change any number.

Protocol per unit (mirrors `adversarial_reactivity_test` with
`degrade_explain_func=True`): the pre-computed attribution slice has a random
`frac` of entries zeroed per image, and — because these metrics re-compute
attributions internally — the explanation function is wrapped so each
invocation zeroes a fresh random `frac` of its output.

In [ ]:
import time, datetime, shutil

CKPT_LOCAL = 'results/ar_v2_units.csv'
CKPT_DRIVE = f'{DRIVE_THESIS}/results/checkpoints/ar_v2_units.csv'
SYNC_SECONDS = 120
SLOW_UNIT_S  = 60
os.makedirs('results', exist_ok=True)

UNIT_COLS = ['model', 'fae_method', 'metric', 'level_idx', 'level_frac',
             'batch_idx', 'n', 'mean_score', 'raw_scores', 'runtime_seconds',
             'timestamp']

def _sync_to_drive():
    try:
        shutil.copy2(CKPT_LOCAL, CKPT_DRIVE)
    except Exception as exc:
        print(f'  WARN: Drive sync failed (continuing): {type(exc).__name__}: {exc}')

# Restore-on-start (fresh runtime): pull the Drive snapshot back.
if not os.path.exists(CKPT_LOCAL) and os.path.exists(CKPT_DRIVE):
    shutil.copy2(CKPT_DRIVE, CKPT_LOCAL)
    print(f'Restored {CKPT_LOCAL} from Drive checkpoint.')

done_units = set()
if os.path.exists(CKPT_LOCAL):
    # on_bad_lines='skip': a runtime that died mid-append can leave one torn
    # line; skipping it just re-runs that single unit.
    _prev = pd.read_csv(CKPT_LOCAL, on_bad_lines='skip').dropna(subset=['batch_idx'])
    done_units = set((r.model, r.fae_method, r.metric,
                      int(r.level_idx), int(r.batch_idx))
                     for r in _prev.itertuples())
    print(f'Resuming: {len(done_units)} units already checkpointed.')

MODEL_NAMES = list(models)
METRIC_NAMES = list(metric_fns)
work = [(m, f, k, li, bi)
        for m in MODEL_NAMES for f in FAE_NAMES for k in METRIC_NAMES
        for li in range(N_LEVELS) for bi in range(N_BATCHES)]
work = [w for w in work if w not in done_units]
print(f'{len(work)} units to run.')

# Base attributions are computed once per (model, FAE) and reused across all
# levels/batches of that pair; the cache holds one pair at a time (~38 MB).
_attr_cache = {}
def base_attributions(model_name, fae_name):
    key = (model_name, fae_name)
    if key not in _attr_cache:
        fn = fae_funcs[model_name][fae_name]
        outs = [np.asarray(fn(models[model_name], x_all[i:i+MINI_BATCH],
                               y_all[i:i+MINI_BATCH]))
                for i in range(0, N_IMAGES, MINI_BATCH)]
        _attr_cache.clear()          # keep memory flat: one pair at a time
        _attr_cache[key] = np.concatenate(outs, axis=0).astype(np.float32)
    return _attr_cache[key]

def degrade_batch(a, frac, rng):
    """Zero a random *frac* of each sample's attribution (AR array degradation)."""
    if frac <= 0.0:
        return a.copy()
    a = a.copy()
    flat = a.reshape(a.shape[0], -1)
    n_total = flat.shape[1]
    n_zero = max(1, int(round(frac * n_total)))
    for b in range(flat.shape[0]):
        idx = rng.choice(n_total, size=n_zero, replace=False)
        flat[b, idx] = 0.0
    return flat.reshape(a.shape)

t_start, t_sync = time.time(), time.time()
for u, (model_name, fae_name, metric_name, li, bi) in enumerate(work, 1):
    t0 = time.perf_counter()
    frac = LEVELS[li]
    sl = slice(bi * MINI_BATCH, (bi + 1) * MINI_BATCH)

    # Restart-invariant per-unit RNG.
    rng = np.random.default_rng([SEED, MODEL_NAMES.index(model_name),
                                 FAE_NAMES.index(fae_name),
                                 METRIC_NAMES.index(metric_name), li, bi])

    a_base = base_attributions(model_name, fae_name)
    a_deg  = degrade_batch(a_base[sl], frac, rng)

    explain_fn = fae_funcs[model_name][fae_name]
    level_explain = (make_degraded_explain_func(explain_fn, frac, rng)
                     if frac > 0.0 else explain_fn)

    mean_score, raw = _call_metric(
        metric_fns[metric_name], models[model_name],
        x_all[sl], y_all[sl], a_deg, device,
        explain_func=level_explain,
        return_raw=True,
    )
    elapsed = time.perf_counter() - t0

    row = pd.DataFrame([{
        'model': model_name, 'fae_method': fae_name, 'metric': metric_name,
        'level_idx': li, 'level_frac': round(frac, 4), 'batch_idx': bi,
        'n': len(raw), 'mean_score': mean_score,
        'raw_scores': ';'.join(f'{s:.6g}' for s in raw),
        'runtime_seconds': round(elapsed, 2),
        'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    }], columns=UNIT_COLS)
    row.to_csv(CKPT_LOCAL, mode='a', header=not os.path.exists(CKPT_LOCAL), index=False)

    if (time.time() - t_sync) >= SYNC_SECONDS or elapsed >= SLOW_UNIT_S:
        _sync_to_drive()
        t_sync = time.time()

    rate = (time.time() - t_start) / u
    eta_h = (len(work) - u) * rate / 3600
    print(f'[{u:4d}/{len(work)}] {model_name:10s} {fae_name:20s} '
          f'{metric_name:16s} lvl={frac:.1f} b={bi} '
          f'mean={mean_score:.4g} {elapsed:.0f}s | ETA {eta_h:.1f}h')

_sync_to_drive()
print(f'\nAll units complete. Total this session: {(time.time()-t_start)/60:.1f} min')

## 8. Aggregate units -> per-cell AR scores (v2)

Pools the 8 mini-batches per level into a level mean, applies the same
zero-variance guard as `adversarial_reactivity_test`, computes the **signed**
Spearman monotonicity over the 10 level means, and reports both the signed
value and `ar_score = |rho|`. The signed column is the direction audit the
thesis review asked for: `expected_sign = -METRIC_DIRECTIONS[metric]`
(a higher-is-better metric should *fall* under degradation, a lower-is-better
metric should *rise*); rows where a strong |rho| has the wrong sign are
flagged — they would indicate AR credit for wrong-direction reactivity.

In [ ]:
from scipy.stats import spearmanr
from src.aggregation.normalize import METRIC_DIRECTIONS

units = pd.read_csv(CKPT_LOCAL, on_bad_lines='skip').dropna(subset=['batch_idx'])
units = units.drop_duplicates(['model', 'fae_method', 'metric',
                               'level_idx', 'batch_idx'], keep='last')
units['level_idx'] = units['level_idx'].astype(int)
units['batch_idx'] = units['batch_idx'].astype(int)
expected_total = len(models) * len(FAE_NAMES) * len(metric_fns) * N_LEVELS * N_BATCHES
assert len(units) >= expected_total, (
    f'{len(units)} units found, {expected_total} expected — run cell 7 to completion first.')

def _parse(s):
    return np.array([float(x) for x in str(s).split(';')])

rows = []
for (model_name, fae_name, metric_name), g in units.groupby(
        ['model', 'fae_method', 'metric']):
    per_level = {li: np.concatenate([_parse(s) for s in d.raw_scores])
                 for li, d in g.groupby('level_idx')}

    # Zero-variance guard on the undegraded (level 0) scores.
    base = per_level[0]
    base = base[~np.isnan(base)]
    if len(base) >= 2 and float(np.std(base)) < 1e-8:
        mono, ar = float('nan'), float('nan')
        level_means = [float('nan')] * N_LEVELS
    else:
        level_means = [float(np.nanmean(per_level[li])) if li in per_level
                       else float('nan') for li in range(N_LEVELS)]
        valid = [(lv, s) for lv, s in zip(LEVELS, level_means) if not np.isnan(s)]
        if len(valid) >= 2 and np.std([s for _, s in valid]) > 1e-12:
            mono = float(spearmanr([lv for lv, _ in valid],
                                   [s for _, s in valid]).statistic)
        elif len(valid) >= 2:
            mono = 0.0
        else:
            mono = float('nan')
        ar = abs(mono) if not np.isnan(mono) else float('nan')

    exp_sign = -METRIC_DIRECTIONS[metric_name]
    wrong = (not np.isnan(mono)) and abs(mono) >= 0.5 and np.sign(mono) != exp_sign
    rows.append({'model': model_name, 'fae_method': fae_name, 'metric': metric_name,
                 'ar_score': ar, 'monotonicity': mono,
                 'expected_sign': int(exp_sign), 'wrong_direction': wrong,
                 'n_images': N_IMAGES, 'n_levels': N_LEVELS,
                 'scores': ';'.join(f'{s:.6g}' for s in level_means),
                 'runtime_seconds': round(g.runtime_seconds.sum(), 1)})

v2 = pd.DataFrame(rows).sort_values(['metric', 'model', 'fae_method'])
OUT_LOCAL = 'results/ar_completion_v2_n64.csv'
v2.to_csv(OUT_LOCAL, index=False)
shutil.copy2(OUT_LOCAL, f'{DRIVE_THESIS}/results/ar_completion_v2_n64.csv')
print(f'Wrote {OUT_LOCAL} (+ Drive copy).\n')
print(v2[['model', 'fae_method', 'metric', 'ar_score', 'monotonicity',
          'wrong_direction']].to_string(index=False))
n_wrong = int(v2.wrong_direction.sum())
print(f'\nDirection audit: {n_wrong} cell(s) with strong wrong-direction reactivity'
      + (' — quote these in the ch5 threat discussion!' if n_wrong else ' — clean.'))
print('\nPer-metric mean AR (v2 vs v1):')
v1 = pd.read_csv(f'{DRIVE_THESIS}/results/ar_completion_progress.csv')
for met in v2.metric.unique():
    print(f'  {met:16s} v2={v2[v2.metric==met].ar_score.mean():.3f} '
          f'v1={v1[v1.metric==met].ar_score.mean():.3f}')

## 9. Patch the Reliability CSV and Save to Drive

Backs up the current CSV to `meta_evaluation_reliability_pre_ARv2.csv`, then
updates `ar_score` and `combined_reliability` (= 0.5 x (existing NR + new AR))
for the recomputed rows only. Metrics not rerun here (e.g. `avg_sensitivity`
unless enabled) keep their v1 values — note that in the thesis if reported.

In [ ]:
REL_CSV = f'{THESIS_ROOT}/results/meta_evaluation_reliability.csv'
rel = pd.read_csv(REL_CSV)

backup = f'{DRIVE_THESIS}/results/meta_evaluation_reliability_pre_ARv2.csv'
if not os.path.exists(backup):
    shutil.copy2(REL_CSV, backup)
    print(f'Backup of pre-ARv2 CSV -> {backup}')

patched = 0
for r in v2.itertuples():
    m = ((rel.model == r.model) & (rel.fae_method == r.fae_method)
         & (rel.metric == r.metric))
    assert m.sum() == 1, f'Expected exactly 1 row for {(r.model, r.fae_method, r.metric)}'
    idx = rel.index[m][0]
    nr = float(rel.at[idx, 'nr_score'])
    rel.at[idx, 'ar_score'] = r.ar_score
    rel.at[idx, 'combined_reliability'] = (
        0.5 * (nr + r.ar_score)
        if not (np.isnan(nr) or np.isnan(r.ar_score)) else float('nan'))
    patched += 1

print(f'Patched {patched} rows.')
print('Valid combined_reliability:',
      rel.combined_reliability.notna().sum(), '/', len(rel))

out_local = 'results/meta_evaluation_reliability.csv'
rel.to_csv(out_local, index=False)
out_drive = f'{DRIVE_THESIS}/results/meta_evaluation_reliability.csv'
shutil.copy2(out_local, out_drive)
print(f'Saved patched CSV -> {out_local} and {out_drive}')

print('''
NEXT STEPS (back on the local machine):
  1. Copy the artifacts into the repo:
       cp <Drive>/thesis/results/meta_evaluation_reliability.csv results/
       cp <Drive>/thesis/results/ar_completion_v2_n64.csv        results/
       cp <Drive>/thesis/results/checkpoints/ar_v2_units.csv     results/
  2. Re-run the local aggregation chain (lambda and the MQ-discount column change):
       .venv/bin/python experiments/compare_rankings.py \\
           --slice-csv ../results/full_run_7fae_12metrics_600.csv \\
           --reliability-csv ../results/meta_evaluation_reliability.csv \\
           --output-csv ../results/ranking_comparison.csv
       .venv/bin/python experiments/run_statistical_analysis.py \\
           --ranking-csv ../results/ranking_comparison.csv \\
           --tests-out ../results/statistical_tests.csv \\
           --cd-out ../results/cd_summary.csv
       .venv/bin/python experiments/make_figures.py
       .venv/bin/python experiments/render_results_tables.py \\
           --n-images-label "full run, 600 images"
  3. Reconcile the prose against the regenerated tables:
       - ch4 sec:exp-meta reliability values + tab:meta caption (state n=64
         for NR and AR, 10 AR levels for the two extended-protocol metrics)
       - ch4/ch5/ch6 MQ-discount numbers (rho AW-vs-MQ, LRP position, |Delta|)
       - abstract + Streszczenie reliability sentence
       - memory: pixel-flipping-direction-bug.md headline numbers
  4. Rebuild praca.pdf and re-audit every changed number.''')

## 10. Optional (~1 min): test accuracy and misclassified fraction

Independent of the AR run — can be executed any time after cell 6. Fills the
“test accuracy: n/a” cells of `tab:model-summary` and quantifies the ch5
target-class caveat (attributions explain the *predicted* class, so on
misclassified images the Localization masks are scored against the wrong
decision). Writes `results/test_accuracy.csv` to Drive.

In [ ]:
import numpy as np, torch, pandas as pd, shutil

acc_rows = []
_BS = 64
_full = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=False)
print(f'{len(_full)} test images')
_labels, _preds = [], {m: [] for m in models}
for _i in range(0, len(_full), _BS):
    batch = [_full[j] for j in range(_i, min(_i + _BS, len(_full)))]
    xb = torch.stack([b['image'] for b in batch]).to(device)
    _labels.extend(int(b['label']) for b in batch)
    with torch.no_grad():
        for m, mdl in models.items():
            _preds[m].extend(mdl(xb).argmax(dim=1).tolist())

_labels = np.array(_labels)
for m in models:
    p = np.array(_preds[m])
    acc = float((p == _labels).mean())
    per_class = {c: float((p[_labels == ci] == ci).mean())
                 for ci, c in enumerate(['melanoma', 'nevus', 'seborrheic_keratosis'])}
    acc_rows.append({'model': m, 'test_accuracy': round(acc, 4),
                     'misclassified_fraction': round(1 - acc, 4),
                     'n': len(_labels), **{f'recall_{k}': round(v, 4)
                                           for k, v in per_class.items()}})
    print(f'{m}: test acc {acc:.1%} | misclassified {1-acc:.1%} | recalls {per_class}')

_both_wrong = float(((np.array(_preds['resnet18']) != _labels)
                     & (np.array(_preds['squeezenet']) != _labels)).mean())
print(f'misclassified by BOTH models: {_both_wrong:.1%}')

_df = pd.DataFrame(acc_rows)
_df.to_csv('results/test_accuracy.csv', index=False)
shutil.copy2('results/test_accuracy.csv', f'{DRIVE_THESIS}/results/test_accuracy.csv')
print('wrote results/test_accuracy.csv (+ Drive copy)')
print('THESIS EDITS: tab:model-summary test-accuracy cells (ch4) and the')
print('ch5 target-class caveat (“a non-trivial fraction” -> the exact number).')